<a href="https://colab.research.google.com/github/pradeep-kottana/Mathematical-foundations-to-Generative-AI/blob/main/RNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
# Sequence Tensor Shape
# For batch_first = True
# batch_size x sequence_length x input_dim

batch_size = 4
sequence_length = 5
input_dim = 3

x = torch.randn(batch_size, sequence_length, input_dim)

print("Sequence input shape:", x.shape)

Sequence input shape: torch.Size([4, 5, 3])


In [3]:
# Simple RNN Layer

rnn = nn.RNN(
    input_size=3,
    hidden_size=8,
    num_layers=1,
    batch_first=True
)

x = torch.randn(4, 5, 3)

output, hidden = rnn(x)

print("Output shape:", output.shape)
print("Hidden shape:", hidden.shape)


Output shape: torch.Size([4, 5, 8])
Hidden shape: torch.Size([1, 4, 8])


In [4]:
# Simple LSTM layer

lstm = nn.LSTM(
    input_size=3,
    hidden_size=8,
    num_layers=1,
    batch_first=True
)

x = torch.randn(4, 5, 3)

output, (hidden, cell) = lstm(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Hidden shape:", hidden.shape)
print("Cell shape:", cell.shape)

Input shape: torch.Size([4, 5, 3])
Output shape: torch.Size([4, 5, 8])
Hidden shape: torch.Size([1, 4, 8])
Cell shape: torch.Size([1, 4, 8])


In [5]:
# GRU Layer

gru = nn.GRU(
    input_size=3,
    hidden_size=8,
    num_layers=1,
    batch_first=True
)

x = torch.randn(4, 5, 3)

output, hidden = gru(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Hidden shape:", hidden.shape)

Input shape: torch.Size([4, 5, 3])
Output shape: torch.Size([4, 5, 8])
Hidden shape: torch.Size([1, 4, 8])


In [6]:
# LSTM Classifier

class LSTMClassifier(nn.Module):

  def __init__(self, input_dim, hidden_dim, num_classes):
    super().__init__()

    self.lstm = nn.LSTM(
        input_size=input_dim,
        hidden_size=hidden_dim,
        num_layers=1,
        batch_first=True
    )

    self.fc = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    output, (hidden, cell) = self.lstm(x)

    # hidden shape:
    # num_layers x batch_size x hidden_dim

    last_hidden = hidden[-1]

    logits = self.fc(last_hidden)

    return logits

model = LSTMClassifier(
    input_dim=3,
    hidden_dim=16,
    num_classes=2
)

x = torch.randn(8, 5, 3)

output = model(x)

print("Output shape:", output.shape)

Output shape: torch.Size([8, 2])


In [7]:
# Toy Sequence Dataset
# Task:
# If sum of sequence values > 0, class 1
# Else class 0

class SequenceDataset(Dataset):

  def __init__(self, num_samples=1000, sequence_length=10, input_dim=3):

    self.X = torch.randn(num_samples, sequence_length, input_dim)

    sums = self.X.sum(dim=(1,2))

    self.y = (sums > 0).long()

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

sequence_dataset = SequenceDataset()

sequence_loader = DataLoader(
    sequence_dataset,
    batch_size=32,
    shuffle=True
)

x_batch, y_batch = next(iter(sequence_loader))

print("Batch input shape:", x_batch.shape)
print("Batch output shape:", y_batch.shape)


Batch input shape: torch.Size([32, 10, 3])
Batch output shape: torch.Size([32])


In [9]:
# Train LSTM Classifier

device = "cuda" if torch.cuda.is_available() else "cpu"

model = LSTMClassifier(
    input_dim=3,
    hidden_dim=32,
    num_classes=2,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):

  model.train()

  total_loss = 0
  correct = 0
  total = 0

  for batch_x, batch_y in sequence_loader:

    batch_x = batch_x.to(device)
    batch_y = batch_y.to(device)

    logits = model(batch_x)
    loss = criterion(logits, batch_y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    predictions = torch.argmax(logits, dim=1)

    correct += (predictions == batch_y).sum().item()
    total += batch_y.size(0)

  acc = correct/total
  print(f"Epoch {epoch}, Loss:{total_loss}, Accuracy:{acc}")


Epoch 0, Loss:21.877052664756775, Accuracy:0.608
Epoch 1, Loss:18.380173921585083, Accuracy:0.848
Epoch 2, Loss:8.89278231561184, Accuracy:0.899
Epoch 3, Loss:6.291296139359474, Accuracy:0.934
Epoch 4, Loss:5.169242307543755, Accuracy:0.948
Epoch 5, Loss:4.566148824989796, Accuracy:0.952
Epoch 6, Loss:4.136970557272434, Accuracy:0.953
Epoch 7, Loss:3.5155270732939243, Accuracy:0.956
Epoch 8, Loss:3.24617163464427, Accuracy:0.967
Epoch 9, Loss:2.900128211826086, Accuracy:0.966


In [10]:
# Save Model

torch.save(model.state_dict(), "lstm_classifier.pth")

print("Model saved successfully.")

Model saved successfully.


In [11]:
# Load Model

loaded_model = LSTMClassifier(
    input_dim=3,
    hidden_dim=32,
    num_classes=2
).to(device)

loaded_model.load_state_dict(torch.load("lstm_classifier.pth", map_location=device))

loaded_model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [12]:
# Reusable Training Function

def train_one_epoch(model, dataloader, criterion, optimizer, device):

  model.train()

  total_loss = 0
  correct = 0
  total = 0

  for inputs, targets in dataloader:
    inputs = inputs.to(device)
    targets = targets.to(device)

    outputs = model(inputs)
    loss = criterion(outputs, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = torch.argmax(outputs, dim=1)
    correct += (predictions == targets).sum().item()
    total += targets.size(0)

  avg_loss = total_loss / len(dataloader)
  accuracy = correct/total

  return avg_loss, accuracy


In [13]:
# Reusable Evaluation Function

def evaluate(model, dataloader, criterion, device):

  model.eval()

  total_loss = 0
  correct = 0
  total = 0

  with torch.no_grad():

    for inputs, targets in dataloader:

      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)
      loss = criterion(outputs, targets)

      total_loss += loss.item()

      predictions = torch.argmax(outputs, dim=1)
      correct += (predictions == targets).sum().item()
      total += targets.size(0)

  avg_loss = total_loss / len(dataloader)
  accuracy = correct/total

  return avg_loss, accuracy

